In [1]:
# ==============================================================================
# Reproducibility Note: To comply with strict reproducibility standards, a fixed
# random seed (random_state=42) is globally enforced across all operations.
# ==============================================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
from sklearn.preprocessing import label_binarize

# Enforce global seed for absolute determinism
np.random.seed(42)

# ==============================================================================
# STEP 1: LOAD DATASET
# ==============================================================================
# Relative path ensures universal compatibility
data_path = "My dataset with class and without missing values.csv"
df = pd.read_csv(data_path)
print("Dataset Shape:", df.shape)

# ==============================================================================
# STEP 2: TARGET ENCODING
# ==============================================================================
target_column = 'mag class'
le = LabelEncoder()
y = le.fit_transform(df[target_column])
class_names = le.classes_.tolist()
print("\nClasses:")
print(class_names)

# ==============================================================================
# STEP 3: DEFINE SEVERE CLASSES
# ==============================================================================
severe_indices = [
    class_names.index(c) for c in class_names if c in ['Strong', 'Major']
]

# ==============================================================================
# STEP 4: USE ALL FEATURES AVAILABLE
# ==============================================================================
# Remove target column to isolate features
X = df.drop(columns=[target_column])

# Store original feature count before encoding
original_feature_count = X.shape[1]
print("\nTotal Original Features:", original_feature_count)

# ==============================================================================
# STEP 5: HANDLE CATEGORICAL FEATURES
# ==============================================================================
categorical_cols = X.select_dtypes(include=['object']).columns
print("\nCategorical Columns:")
print(categorical_cols.tolist())

# One-hot encoding
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
print("\nFeature Shape After Encoding:", X.shape)

# ==============================================================================
# STEP 6: TRAIN-TEST SPLIT
# ==============================================================================
# 70:30 Stratified split using the fixed random seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# ==============================================================================
# STEP 7: FEATURE SCALING
# ==============================================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==============================================================================
# STEP 8: TRAIN KNN CLASSIFIER (Proxy Benchmark)
# ==============================================================================
knn_model = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn_model.fit(X_train_scaled, y_train)

# ==============================================================================
# STEP 9: PREDICTIONS
# ==============================================================================
y_pred = knn_model.predict(X_test_scaled)
y_prob = knn_model.predict_proba(X_test_scaled)

# ==============================================================================
# STEP 10: METRICS
# ==============================================================================
accuracy = accuracy_score(y_test, y_pred) * 100
precision = precision_score(y_test, y_pred, average='weighted') * 100
f1 = f1_score(y_test, y_pred, average='weighted') * 100

# Recall for severe classes only
recalls = recall_score(y_test, y_pred, average=None)
severe_recall = np.mean([recalls[idx] for idx in severe_indices]) * 100

# ROC-AUC
y_test_bin = label_binarize(y_test, classes=np.arange(len(class_names)))
roc_auc = roc_auc_score(y_test_bin, y_prob, average='macro', multi_class='ovr')

# ==============================================================================
# STEP 11: CREATE FINAL TABLE
# ==============================================================================
results_df = pd.DataFrame({
    'Feature Strategy': ['Full Feature Set'],
    'Dimension (d)': [original_feature_count],
    'Accuracy (%)': [round(accuracy, 2)],
    'Precision (%)': [round(precision, 2)],
    'Recall (Severe Class) (%)': [round(severe_recall, 2)],
    'F1-Score (%)': [round(f1, 2)],
    'ROC-AUC': [round(roc_auc, 3)]
})

# ==============================================================================
# STEP 12: DISPLAY RESULTS
# ==============================================================================
print("\n")
print("=" * 100)
print("TABLE 2 FULL FEATURE SET RESULTS")
print("=" * 100)
print(results_df.to_string(index=False))

# ==============================================================================
# STEP 13: EXPORT CSV
# ==============================================================================
results_df.to_csv("Full_Feature_Set_Table2.csv", index=False)
print("\nSaved as: Full_Feature_Set_Table2.csv")

Dataset Shape: (19402, 17)

Classes:
['Major', 'Strong', 'light', 'moderate']

Total Original Features: 16

Categorical Columns:
['magType']

Feature Shape After Encoding: (19402, 22)


TABLE 2 FULL FEATURE SET RESULTS
Feature Strategy  Dimension (d)  Accuracy (%)  Precision (%)  Recall (Severe Class) (%)  F1-Score (%)  ROC-AUC
Full Feature Set             16         93.59          93.12                      33.21         93.15    0.933

Saved as: Full_Feature_Set_Table2.csv
